In [63]:
import pandas as pd
import json
import wn
import sys
sys.path.append('../..')
from algo import utils

In [64]:
file = "probas.csv"
to_file = "probas200.csv"
id_to_label_file = "id_to_label.json"
id_to_label_new_file = "id_to_label200.json"
lexicon_original = "omw-subset-en"
lexicon_name = "omw-subset200-en"

In [65]:
df = pd.read_csv(file)
labels = df['label'].unique()

with open(id_to_label_file) as f:
    id_to_label = json.load(f)

node_to_id = {}
id_to_node = {}
with open(id_to_label_file, encoding='utf-8') as f:
        string_mapping = json.load(f)
        for i, string_node in enumerate(string_mapping):
            synset = wn.synset(string_node)
            node_to_id[synset] = i
            id_to_node[i] = synset

In [66]:
label_synsets = [id_to_node[idx] for idx in labels]

def replace(input, replace, with_):
    input.replace(replace, with_)

hypernym_closure = set(utils.flatten([utils.hypernym_paths_flat(node) for node in label_synsets]) + label_synsets)
len(hypernym_closure)
    

552

In [78]:
for synset in hypernym_closure:
    print(f"""
<Synset id="{synset.id.replace(lexicon_original, lexicon_name)}" ili="{synset.ili.id.replace(lexicon_original, lexicon_name)}" partOfSpeech="n">
    <Definition>{synset.definition()}</Definition>
""")
    for hypernym in synset.hypernyms():
        if hypernym in hypernym_closure:
            print(f'    <SynsetRelation target="{hypernym.id.replace(lexicon_original, lexicon_name)}" relType="hypernym"/>')
    for hyponym in synset.hyponyms():
        # if hypernym not in 
        if hyponym.id != "*INFERRED*" and hyponym in hypernym_closure:
            print(f'    <SynsetRelation target="{hyponym.id.replace(lexicon_original, lexicon_name)}" relType="hyponym"/>')
    print("</Synset>")


<Synset id="omw-subset200-en-02159955-n" ili="i46823" partOfSpeech="n">
    <Definition>small air-breathing arthropod</Definition>

    <SynsetRelation target="omw-subset200-en-01767661-n" relType="hypernym"/>
    <SynsetRelation target="omw-subset200-en-02164464-n" relType="hyponym"/>
    <SynsetRelation target="omw-subset200-en-02274024-n" relType="hyponym"/>
</Synset>

<Synset id="omw-subset200-en-13040303-n" ili="i105258" partOfSpeech="n">
    <Definition>any of various ill-smelling brown-capped fungi of the order Phallales</Definition>

    <SynsetRelation target="omw-subset200-en-12992868-n" relType="hypernym"/>
</Synset>

<Synset id="omw-subset200-en-01525720-n" ili="i43268" partOfSpeech="n">
    <Definition>passerine bird having specialized vocal apparatus</Definition>

    <SynsetRelation target="omw-subset200-en-01524359-n" relType="hypernym"/>
    <SynsetRelation target="omw-subset200-en-01529672-n" relType="hyponym"/>
</Synset>

<Synset id="omw-subset200-en-01770393-n" ili

In [68]:
label_cols = [f'label_{l}' for l in labels]
cols_to_keep = ['id', 'content', 'label', ] + label_cols

label_sum = df[label_cols].sum(axis=1)
for col in label_cols:
    df[col] = df[col] / label_sum

df_filtered = df[cols_to_keep]

In [76]:
i = 0
new_i = 0
id_to_label200 = {}
id_to_new_id = {}
for synset_id, label in id_to_label.items():
    if i in labels:
        id_to_label200[synset_id] = id_to_label[synset_id]
        id_to_new_id[i] = new_i
        new_i += 1
    i += 1

with open(id_to_label_new_file, 'w+') as f:
    json.dump(id_to_label200, f)

df_filtered.label = df_filtered.label.map(lambda x: id_to_new_id[x])

/var/folders/74/vwtt1gl561g84w722clplwrr0000gn/T/ipykernel_3345/66984805.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered.label = df_filtered.label.map(lambda x: id_to_new_id[x])


In [77]:
df_filtered.to_csv(to_file)

In [ ]:
root = wordnet.roots